# Simulación de Alta Fidelidad en Tiempo Real: Modelo de Kuznetsov (1992)

Este Jupyter Notebook presenta un análisis avanzado, interactivo y de alta fidelidad para el **modelo de crecimiento tumoral y respuesta inmune de Kuznetsov (1992)**.

El modelo describe la competencia dinámica entre el sistema inmunitario (específicamente, **Células Efectoras $E$** como los linfocitos T citotóxicos) y una población de **Células Tumorales $T$**.

### Ecuaciones del Sistema Dinámico

El sistema de dos ecuaciones diferenciales ordinarias (EDOs) no lineales acopladas está dado por:

$$\frac{dE}{dt} = s + \frac{p E T}{g + T} - m E T - d E$$

$$\frac{dT}{dt} = a T (1 - b T) - n E T$$

#### Interpretación de las Variables y Parámetros:
- **$E(t)$**: Población de células efectoras en el sitio del tumor en el tiempo $t$.
- **$T(t)$**: Población de células tumorales en el tiempo $t$.
- **$s$**: Tasa de producción o flujo basal constante de células efectoras hacia el tejido.
- **$p$**: Tasa de reclutamiento o acumulación de efectoras inducida por la presencia de antígenos tumorales.
- **$g$**: Constante de semisaturación de la respuesta de reclutamiento inmunitario.
- **$m$**: Tasa de inactivación o destrucción de células efectoras por el contacto directo con células tumorales.
- **$d$**: Tasa de muerte natural o migración (decaecimiento) de las células efectoras.
- **$a$**: Tasa máxima de crecimiento intrínseco del tumor.
- **$b$**: Inverso de la capacidad de carga del tejido ($1/b$ representa el tamaño límite que puede alcanzar físicamente el tumor en ausencia de respuesta inmune).
- **$n$**: Tasa de destrucción física (lisis) de células tumorales por la acción citotóxica de las células efectoras.

---


## 1. Configuración de Librerías y Diseño Visual Premium

Cargamos las dependencias de cálculo numérico, integración matemática y animación interactiva. Configuramos un estilo de diseño oscuro y moderno con tipografías legibles y colores vibrantes de tipo HSL.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.integrate import solve_ivp
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
import os

# Estilo de visualización de alta definición (Premium Dark Mode)
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f172a',      # Color de fondo de la figura (Tailwind slate-900)
    'axes.facecolor': '#1e293b',        # Color de fondo de los paneles (Tailwind slate-800)
    'axes.edgecolor': '#475569',        # Color del borde de los ejes
    'grid.color': '#334155',            # Líneas de cuadrícula sutiles
    'grid.linestyle': ':',
    'grid.alpha': 0.5,
    'xtick.color': '#94a3b8',           # Etiquetas de marcas en color suave
    'ytick.color': '#94a3b8',
    'text.color': '#cbd5e1',            # Texto principal
    'axes.labelcolor': '#cbd5e1',
    'axes.titlecolor': '#f8fafc',       # Título principal brillante
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica']
})


## 2. Parámetros del Modelo de Kuznetsov (BCL1)

Definimos las constantes del sistema de acuerdo con las estimaciones experimentales de Kuznetsov et al. (1992) para el linfoma de células B BCL1 en ratones.

In [ ]:
# --- Parámetros de la cepa de linfoma BCL1 ---
a = 0.18            # Tasa max de crecimiento tumoral (dia-1)
b = 2.0e-9          # Capacidad de carga inversa (celulas-1)
s = 1.3e4           # Influx basal de efectoras (celulas/dia)
p = 0.1245          # Tasa de reclutamiento estimulada por tumor (dia-1)
g = 2.019e7         # Semisaturacion de respuesta inmune (celulas)
m = 3.422e-10       # Tasa de inactivacion de efectoras (dia-1 * celulas-1)
n = 1.101e-7        # Tasa de lisis tumoral (dia-1 * celulas-1)
d = 0.0412          # Muerte natural o salida de efectoras (dia-1)

# Definición del sistema de EDOs acopladas
def kuznetsov_ode(t, y):
    E, T = y
    dE_dt = s + (p * E * T) / (g + T) - m * E * T - d * E
    dT_dt = a * T * (1.0 - b * T) - n * E * T
    return [dE_dt, dT_dt]


## 3. Integración Numérica Precisa (RK45)

Precalculamos tres escenarios dinámicos cualitativamente distintos a partir de condiciones iniciales diferentes sobre un horizonte temporal de 150 días:

1. **Región de Dormancia (Control Tumoral):** El tumor es controlado por la respuesta inmune y oscila hacia un equilibrio estable de bajo tamaño ($T \approx 1.2 \times 10^7$ células).
2. **Frontera Crítica (Separatriz/Transición):** La condición inicial se encuentra extremadamente cerca de la frontera crítica del espacio de fases, manteniendo una larga indecisión y oscilación antes de bifurcarse.
3. **Escape Tumoral:** La carga tumoral sobrepasa la capacidad de destrucción del sistema inmune, provocando el crecimiento logístico hasta la máxima capacidad de carga del tejido ($T \approx 5.0 \times 10^8$ células).

In [ ]:
t_span = (0.0, 150.0)
t_eval = np.linspace(t_span[0], t_span[1], 1200) # Alta resolución temporal para suavidad visual

# 1. Caso de Dormancia
y0_dorm = [3.3e5, 1.0e6]  # E0 = 330,000 | T0 = 1,000,000
sol_dorm = solve_ivp(kuznetsov_ode, t_span, y0_dorm, t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)

# 2. Caso de Escape Tumoral
y0_escape = [3.3e5, 5.0e7] # E0 = 330,000 | T0 = 50,000,000
sol_escape = solve_ivp(kuznetsov_ode, t_span, y0_escape, t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)

# 3. Caso de Frontera Crítica
y0_front = [3.3e5, 6.4e6]  # E0 = 330,000 | T0 = 6,400,000 (cerca de la separatrix)
sol_front = solve_ivp(kuznetsov_ode, t_span, y0_front, t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)

print("Integración de trayectorias completada correctamente.")


## 4. Simulación Biofísica 2D basada en Agentes (Física de Partículas)

Implementamos la lógica física que gobernará las células del tejido en el panel derecho. 
Para simular las fluctuaciones de un tejido vivo, las partículas se mueven continuamente mediante un **desplazamiento browniano** (caminata aleatoria gaussiana) en cada frame.

Mapeamos la densidad poblacional de las EDOs continuas (que varían en varios órdenes de magnitud) a recuentos de partículas discretas utilizando una **escala logarítmica**. De este modo, los cambios en poblaciones bajas y altas se aprecian con total claridad sin saturar la pantalla o ralentizar el renderizado.

Cuando la población aumenta, se introducen nuevas células en posiciones aleatorias de la pantalla. Cuando disminuye (debido a lisis o inactivación), se remueven de forma ordenada para mantener la continuidad de las trayectorias de las partículas restantes.

In [ ]:
class CellAgentSimulation:
    def __init__(self, E_traj, T_traj, sigma=0.015):
        self.E_traj = E_traj
        self.T_traj = T_traj
        self.sigma = sigma  # Desviación estándar del paso browniano
        
        # Límites para el mapeo logarítmico a partículas representativas
        self.log_E_min, self.log_E_max = 4.5, 6.5
        self.log_T_min, self.log_T_max = 5.0, 8.7
        
        # Inicialización de las matrices de posiciones de las partículas [x, y]
        self.pos_E = np.empty((0, 2))
        self.pos_T = np.empty((0, 2)) 

    def get_particle_counts(self, E_val, T_val):
        # Convierte los valores continuos de E y T a recuentos discretos de partículas
        log_E = np.log10(np.maximum(E_val, 10.0))
        log_T = np.log10(np.maximum(T_val, 10.0))
        
        # Mapeo lineal del exponente a rango [5, 150] para efectoras y [2, 150] para tumorales
        count_E = int(np.clip(5 + 145 * (log_E - self.log_E_min) / (self.log_E_max - self.log_E_min), 5, 150))
        count_T = int(np.clip(2 + 148 * (log_T - self.log_T_min) / (self.log_T_max - self.log_T_min), 2, 150))
        return count_E, count_T

    def update_frame(self, E_val, T_val):
        target_E, target_T = self.get_particle_counts(E_val, T_val)
        
        # 1. Actualizar posiciones de células Efectoras (E)
        curr_E = len(self.pos_E)
        if target_E > curr_E:
            # Se crean nuevas células inmunitarias en posiciones uniformes al azar
            new_cells = np.random.uniform(0.05, 0.95, (target_E - curr_E, 2))
            self.pos_E = np.vstack([self.pos_E, new_cells]) if curr_E > 0 else new_cells
        elif target_E < curr_E:
            # Se remueven partículas (lisis / decaimiento)
            self.pos_E = self.pos_E[:target_E]
            
        # 2. Actualizar posiciones de células Tumorales (T)
        curr_T = len(self.pos_T)
        if target_T > curr_T:
            new_cells = np.random.uniform(0.05, 0.95, (target_T - curr_T, 2))
            self.pos_T = np.vstack([self.pos_T, new_cells]) if curr_T > 0 else new_cells
        elif target_T < curr_T:
            self.pos_T = self.pos_T[:target_T]
            
        # 3. Aplicar diferencial browniano (caminata física continua)
        if len(self.pos_E) > 0:
            self.pos_E += np.random.normal(0, self.sigma, self.pos_E.shape)
            self.pos_E = np.clip(self.pos_E, 0.02, 0.98) # Delimitar límites rígidos del tejido
            
        if len(self.pos_T) > 0:
            self.pos_T += np.random.normal(0, self.sigma, self.pos_T.shape)
            self.pos_T = np.clip(self.pos_T, 0.02, 0.98)
            
        return self.pos_E, self.pos_T


## 5. Control Interactivo del Escenario de Simulación

A continuación, se define el panel de control. Utilizando **IPython Widgets**, puedes seleccionar qué régimen de la EDO y qué trayectoria activa deseas simular interactuando con el menú desplegable. El gráfico de dos paneles se compilará y renderizará dinámicamente de forma fluida y en tiempo real para el caso seleccionado.

In [ ]:
# Crear el selector desplegable interactivo
selector_caso = widgets.Dropdown(
    options=[
        ('Control Inmunológico (Dormancia)', 'Dormancia'),
        ('Fase Crítica (Frontera de Inestabilidad)', 'Frontera'),
        ('Escape Tumoral Completo', 'Escape')
    ],
    value='Frontera',
    description='Seleccionar Escenario Activo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

# Contenedor de salida para la animación de alta definición
contenedor_animacion = widgets.Output()

def renderizar_caso(caso_activo):
    # Limpiar figuras anteriores para prevenir fugas de memoria
    plt.close('all')
    
    # Mapeo de soluciones según el caso activo
    if caso_activo == 'Dormancia':
        sol_act = sol_dorm
        color_act = '#10b981' # Verde esmeralda
        nombre_regimen = "Control Inmune (Dormancia)"
    elif caso_activo == 'Escape':
        sol_act = sol_escape
        color_act = '#f97316' # Naranja
        nombre_regimen = "Escape Tumoral Completo"
    else:
        sol_act = sol_front
        color_act = '#a855f7' # Púrpura neón
        nombre_regimen = "Fase Crítica (Inestabilidad)"

    t_data = sol_act.t
    E_data = sol_act.y[0]
    T_data = sol_act.y[1]

    # Crear simulación física de partículas para este caso
    sim = CellAgentSimulation(E_data, T_data, sigma=0.014)

    # Construcción de la interfaz de dos paneles
    fig, (ax_phase, ax_agents) = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={'width_ratios': [1.1, 1]})
    fig.patch.set_facecolor('#0f172a')

    # 1. Panel Izquierdo: Plano de Fase
    ax_phase.set_facecolor('#1e293b')
    ax_phase.set_title("Espacio de Fases Continuo (RK45)", pad=15, fontsize=12, fontweight='bold')
    ax_phase.set_xlabel("Células Efectoras (E)", fontsize=10, labelpad=10)
    ax_phase.set_ylabel("Células Tumorales (T)", fontsize=10, labelpad=10)

    # Dibujar las 3 órbitas precalculadas como fondo de referencia sutil
    ax_phase.plot(sol_dorm.y[0], sol_dorm.y[1], color='#10b981', linestyle='--', alpha=0.22, linewidth=1.5, label='Dormancia (Dosis baja)')
    ax_phase.plot(sol_front.y[0], sol_front.y[1], color='#a855f7', linestyle='--', alpha=0.22, linewidth=1.5, label='Frontera Crítica (Transición)')
    ax_phase.plot(sol_escape.y[0], sol_escape.y[1], color='#f97316', linestyle='--', alpha=0.22, linewidth=1.5, label='Escape Tumoral (Dosis alta)')

    # Trayectoria de crecimiento activa
    line_active, = ax_phase.plot([], [], color=color_act, linewidth=3, label=f'Ruta Activa ({caso_activo})')
    head_marker = ax_phase.scatter([], [], color=color_act, s=120, zorder=6, edgecolors='white', linewidths=1.0)

    ax_phase.set_xscale('log')
    ax_phase.set_yscale('log')
    ax_phase.set_xlim(1.5e4, 5e6)
    ax_phase.set_ylim(2e4, 1e9)
    ax_phase.grid(True, which='both', color='#334155', linestyle=':', alpha=0.5)
    ax_phase.legend(loc='lower left', framealpha=0.9, facecolor='#0f172a', edgecolor='#475569', fontsize=9)

    # 2. Panel Derecho: Microambiente Celular 2D
    ax_agents.set_facecolor('#1e293b')
    ax_agents.set_title("Microambiente Tumoral (Física de Agentes 2D)", pad=15, fontsize=12, fontweight='bold')
    ax_agents.set_xlim(0, 1)
    ax_agents.set_ylim(0, 1)
    ax_agents.set_xticks([])
    ax_agents.set_yticks([])

    # Células Efectoras Inmunitarias (Cian)
    glow_E = ax_agents.scatter([], [], color='#38bdf8', s=180, alpha=0.12, zorder=2)
    scatter_E = ax_agents.scatter([], [], color='#38bdf8', s=45, alpha=0.9, edgecolors='white', linewidths=0.5, label='Inmunes (Efectoras)', zorder=3)

    # Células Tumorales de Cáncer (Rosa/Rojo)
    glow_T = ax_agents.scatter([], [], color='#f43f5e', s=180, alpha=0.12, zorder=2)
    scatter_T = ax_agents.scatter([], [], color='#f43f5e', s=45, alpha=0.9, edgecolors='white', linewidths=0.5, label='Tumorales (Cáncer)', zorder=3)

    ax_agents.legend(loc='upper right', framealpha=0.9, facecolor='#0f172a', edgecolor='#475569', fontsize=9)

    for spine in ax_agents.spines.values():
        spine.set_edgecolor('#475569')
        spine.set_linewidth(1.5)

    # Texto de Telemetría Dinámica
    telemetry_txt = ax_agents.text(
        0.03, 0.97, "", transform=ax_agents.transAxes,
        color='#cbd5e1', fontfamily='monospace', fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#0f172a', alpha=0.9, edgecolor='#475569')
)

    # Inicializador de la animación
    def init_anim():
        line_active.set_data([], [])
        head_marker.set_offsets(np.empty((0, 2)))
        scatter_E.set_offsets(np.empty((0, 2)))
        glow_E.set_offsets(np.empty((0, 2)))
        scatter_T.set_offsets(np.empty((0, 2)))
        glow_T.set_offsets(np.empty((0, 2)))
        telemetry_txt.set_text("")
        return line_active, head_marker, scatter_E, glow_E, scatter_T, glow_T, telemetry_txt

    # Reducir número de fotogramas para optimizar velocidad del blitting
    sample_rate = 2 
    frame_indices = range(0, len(t_data), sample_rate)

    def update_anim(frame_idx):
        idx = frame_indices[frame_idx]
        t_curr = t_data[idx]
        E_curr = E_data[idx]
        T_curr = T_data[idx]
        
        # 1. Plano de fases dinámico
        line_active.set_data(E_data[:idx+1], T_data[:idx+1])
        head_marker.set_offsets([[E_curr, T_curr]])
        
        # Pulsación armónica del marcador principal
        pulsation = 110 + 40 * np.sin(frame_idx * 0.35)
        head_marker.set_sizes([pulsation])
        
        # 2. Caminata browniana de los agentes celulares
        pos_E, pos_T = sim.update_frame(E_curr, T_curr)
        
        if len(pos_E) > 0:
            scatter_E.set_offsets(pos_E)
            glow_E.set_offsets(pos_E)
        else:
            scatter_E.set_offsets(np.empty((0, 2)))
            glow_E.set_offsets(np.empty((0, 2)))
            
        if len(pos_T) > 0:
            scatter_T.set_offsets(pos_T)
            glow_T.set_offsets(pos_T)
        else:
            scatter_T.set_offsets(np.empty((0, 2)))
            glow_T.set_offsets(np.empty((0, 2)))
            
        # Diagnóstico biológico dinámico
        if T_curr < 1.3e7:
            diagnostico = "Control Inmune (Dormancia)"
        elif T_curr > 1.5e8:
            diagnostico = "Escape Tumoral Agresivo"
        else:
            diagnostico = "Transición e Inestabilidad"
            
        # Telemetría de texto formateada
        info_string = (
            f" TELEMETRÍA DEL TEJIDO VIVO\n"
            f" ───────────────────────────\n"
            f" Día de Simulación : {t_curr:5.1f} días\n"
            f" Células Inmunes   : {E_curr:5.2e}\n"
            f" Células Tumorales : {T_curr:5.2e}\n"
            f" Relación Inmune/T : {E_curr/T_curr:6.4f}\n"
            f" Estado del Tejido : {diagnostico}\n"
            f" Régimen de EDOs   : {caso_activo}"
        )
        telemetry_txt.set_text(info_string)
        
        return line_active, head_marker, scatter_E, glow_E, scatter_T, glow_T, telemetry_txt

    fps_rate = 30
    interval_ms = 1000 // fps_rate
    total_frames = len(frame_indices)

    anim = animation.FuncAnimation(
        fig, update_anim, init_func=init_anim, 
        frames=total_frames, interval=interval_ms, blit=True
    )
    plt.close(fig)
    return anim

def cambio_seleccion(change):
    with contenedor_animacion:
        clear_output(wait=True)
        print(f"Compilando animación para el caso: '{change.new}'... Por favor, espera unos segundos.")
        anim = renderizar_caso(change.new)
        display(HTML(anim.to_jshtml()))

# Enlazar evento de cambio de selección
selector_caso.observe(cambio_seleccion, names='value')

# Mostrar los controles interactivos
display(selector_caso)
display(contenedor_animacion)

# Lanzamiento y renderizado inicial (caso Frontera)
class ObjetoCambioInicial:
    def __init__(self, valor):
        self.new = valor

cambio_seleccion(ObjetoCambioInicial('Frontera'))
